In [ ]:
import pickle
import numpy as np
import tensorflow as tf

# Path to your data file
data_path = '/content/drive/MyDrive/Colab Notebooks/RML2016.10a_dict.pkl'

# Load data
with open(data_path, 'rb') as file:
    Xd = pickle.load(file, encoding='iso-8859-1')

# Extract mods and snrs
mods, snrs = [sorted(list(set([k[j] for k in Xd.keys()]))) for j in [0,1]]

X = []
lbl = []
train_idx = []
val_idx = []
np.random.seed(2016)
a = 0

# Populate X and lbl
for mod in mods:
    for snr in snrs:
        X.append(Xd[(mod, snr)])
        for i in range(Xd[(mod, snr)].shape[0]):
            lbl.append((mod, snr))
        train_idx += list(np.random.choice(range(a*1000, (a+1)*1000), size=600, replace=False))
        val_idx += list(np.random.choice(list(set(range(a*1000, (a+1)*1000)) - set(train_idx)), size=200, replace=False))
        a += 1

# Stack X
X = np.vstack(X)

# Define test_idx
n_examples = X.shape[0]
test_idx = list(set(range(0, n_examples)) - set(train_idx) - set(val_idx))
np.random.shuffle(train_idx)
np.random.shuffle(val_idx)
np.random.shuffle(test_idx)

# Split into train, val, and test sets
X_train = X[train_idx]
X_val = X[val_idx]
X_test = X[test_idx]

# Define to_onehot function
def to_onehot(yy):
    yy1 = np.zeros([len(yy), len(mods)])
    yy1[np.arange(len(yy)), yy] = 1
    return yy1

# Generate Y_train, Y_val, and Y_test
Y_train = to_onehot(list(map(lambda x: mods.index(lbl[x][0]), train_idx)))
Y_val = to_onehot(list(map(lambda x: mods.index(lbl[x][0]), val_idx)))
Y_test = to_onehot(list(map(lambda x: mods.index(lbl[x][0]), test_idx)))

# Swap axes for training, validation, and test sets
X_train = X_train.swapaxes(2, 1)
X_val = X_val.swapaxes(2, 1)
X_test = X_test.swapaxes(2, 1)



print(X_train.shape)
print(X_val.shape)
print(X_test.shape)
print(Y_train.shape)
print(Y_val.shape)
print(Y_test.shape)

In [ ]:
from tensorflow.keras.layers import Input,GRU, Conv2D, Dense,Add,Subtract, Dropout,Multiply, BatchNormalization, Activation,Lambda, Flatten, Reshape, MaxPooling2D, concatenate
from keras.callbacks import EarlyStopping
X1_train=X_train[:,:,0]
X1_test=X_test[:,:,0]
X1_val=X_val[:,:,0]
X2_train=X_train[:,:,1]
X2_test=X_test[:,:,1]
X2_val=X_val[:,:,1]
X_train=np.expand_dims(X_train,axis=3)
X_test=np.expand_dims(X_test,axis=3)
X_val=np.expand_dims(X_val,axis=3)
print(X_train.shape)

print(X_test.shape)
classes = mods


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import csv

def plot_confusion_matrix(cm, title='Confusion matrix', cmap=plt.get_cmap("PuRd"), labels=[], save_filename=None):
    plt.figure(figsize=(6, 6))
    plt.imshow(cm * 100, interpolation='nearest', cmap=cmap)
    plt.colorbar()
    tick_marks = np.arange(len(labels))
    plt.xticks(tick_marks, labels, rotation=90, size=12)
    plt.yticks(tick_marks, labels, size=12)

    for i in range(len(tick_marks)):
        for j in range(len(tick_marks)):
            value = int(np.around(cm[i, j] * 100))
            if i != j:
                text = plt.text(j, i, f"{value}%", ha="center", va="center", fontsize=10)
            elif i == j:
                if value == 100:
                    text = plt.text(j, i, f"{value}%", ha="center", va="center", fontsize=7, color="white")
                else:
                    text = plt.text(j, i, f"{value}%", ha="center", va="center", fontsize=10, color="white")

    plt.tight_layout()
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')



In [ ]:
def calculate_confusion_matrix(Y,Y_hat,classes):
    n_classes = len(classes)
    conf = np.zeros([n_classes,n_classes])
    confnorm = np.zeros([n_classes,n_classes])

    for k in range(0,Y.shape[0]):
        i = list(Y[k,:]).index(1)
        j = int(np.argmax(Y_hat[k,:]))
        conf[i,j] = conf[i,j] + 1

    for i in range(0,n_classes):
        confnorm[i,:] = conf[i,:] / np.sum(conf[i,:])

    right = np.sum(np.diag(conf))
    wrong = np.sum(conf) - right
    return confnorm,right,wrong

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.layers import Conv2D,Input, MaxPooling2D, Flatten, Dense, Dropout, Reshape,GRU,LSTM
from tensorflow.keras.optimizers import Adam
def CNN(input_shape=(128, 2), input_shape2=(128, 1), classes=11):

    # Define input layers
    dr = 0.5 # dropout rate (%)
    input = Input(shape=input_shape + (1,), name='input1')  # Shape will be (None, 2, 128, 1)
    input1 = Input(input_shape2, name='input2')
    input2 = Input(input_shape2, name='input3')



    # spatial feature
    x3 = Conv2D(75, (8,2), padding='valid', activation="relu", name="conv1_1", kernel_initializer='glorot_uniform')(
        input)
    x3 = Conv2D(25, (5,1), padding='valid', activation="relu", name="conv1_2", kernel_initializer='glorot_uniform')(
        x3)
    # temporal feature
    x4 = Reshape(target_shape=((117,25)), name='reshape4')(x3)
    x4= GRU(units=128)(x4)

    x = Dense(classes, activation='softmax', name='softmax')(x4)

    model = Model(inputs = [input,input1,input2], outputs=x)



    return model

model=CNN()
model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='adam')
model.summary()

modtype=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM']


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import csv
model.load_weights('/content/drive/MyDrive/Colab Notebooks/CNN_2D_weights.h5')
classes=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM']
plt.figure(figsize=(12, 10))  # Example size, adjust as needed
test_Y_hat = model.predict([X_test, X1_test, X2_test],)
confnorm, _, _ = calculate_confusion_matrix(Y_test, test_Y_hat, classes)
plot_confusion_matrix(confnorm, labels=classes)
acc = {}
acc_mod_snr = np.zeros( (len(classes),len(snrs)) )
i = 0
for snr in snrs:
        # Extract classes @ SNR
        test_SNRs = [lbl[x][1] for x in test_idx]
        test_X_i=X_test[np.where(np.array(test_SNRs) == snr)]
        test_X1_i = X1_test[np.where(np.array(test_SNRs) == snr)]
        test_X2_i = X2_test[np.where(np.array(test_SNRs) == snr)]
        test_Y_i = Y_test[np.where(np.array(test_SNRs) == snr)]
        test_Y_i_hat = model.predict([test_X_i,test_X1_i,test_X2_i])

        confnorm_i,cor,ncor = calculate_confusion_matrix(test_Y_i,test_Y_i_hat,classes)
        acc[snr] = 1.0 * cor / (cor + ncor)
        result = cor / (cor + ncor)

        plot_confusion_matrix(confnorm_i, labels=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM'])
        plt.show()
        plt.savefig('figure3.png')
        acc_mod_snr[:,i] = np.round(np.diag(confnorm_i)/np.sum(confnorm_i,axis=1),3)
        i = i +1


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

num_mod = 11

plt.figure(figsize=(8, 6))
plt.xlabel("Signal to Noise Ratio")
plt.ylabel("Classification Accuracy")
plt.title("Classification Accuracy for Each Mod")

for i in range(0, num_mod):
    plt.plot(snrs, acc_mod_snr[i], label=classes[i])

# Add legend, grid, save, and show plot
plt.legend()
plt.grid()
plt.savefig('marah.png')
plt.show()
plt.close()


In [ ]:
# Plot Accuracy vs. SNR
plt.figure(figsize=(10, 6))
plt.plot(snrs, list(map(lambda x: acc[x], snrs)), marker='o', linestyle='-', color='b', label='Accuracy')

for snr, accuracy in zip(snrs, list(map(lambda x: acc[x], snrs))):
    plt.annotate(f'{accuracy:.2f}',
                 (snr, accuracy),
                 textcoords="offset points",
                 xytext=(0,10),
                 ha='center')

plt.xlabel("Signal to Noise Ratio (SNR)")
plt.ylabel("Classification Accuracy")
plt.title("Classification Accuracy vs. SNR")
plt.grid(True)
plt.savefig('figure1.png')
plt.show()

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Activation,
    Lambda,
    Multiply,
    Add,
    Subtract,
    Reshape,
    concatenate,
    Conv1D,
    MaxPooling1D,
    Flatten,
)

def CNN(input_shape=(128, 1), num_classes=11):
    # Define inputs
    input1 = Input(shape=input_shape, name='input1')
    input2 = Input(shape=input_shape, name='input2')
    input=concatenate([input1,input2])
    x=Conv1D(64, kernel_size=3 ,activation='relu')(input)
    x=MaxPooling1D(pool_size=2)(x)
    x=Dropout(0.3)(x)
    x=Conv1D(128, kernel_size=3, activation='relu')(x)
    x=MaxPooling1D(pool_size=2)(x)
    x=Dropout(0.3)(x)
    x=Conv1D(256, kernel_size=3, activation='relu')(x)
    x=MaxPooling1D(pool_size=2)(x)
    x=Dropout(0.3)(x)
    # Flatten the output
    x = Flatten()(x)
    # Dense layer with 512 neurons
    x = Dense(512, activation='relu')(x)
    x= Dropout(0.3)(x)
    # Output layer with softmax activation
    output = Dense(num_classes, activation='softmax')(x)

    # Create model
    model = Model(inputs=[input1, input2], outputs=output)

    return model

# Example usage
model4= CNN()
model4.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='adam')
model4.summary()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import csv
model4.load_weights('/content/drive/MyDrive/Colab Notebooks/marahfinalCNN.h5')
classes=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM']
plt.figure(figsize=(12, 10))  # Example size, adjust as needed
test_Y_hat = model4.predict([X1_test, X2_test],)
confnorm, _, _ = calculate_confusion_matrix(Y_test, test_Y_hat, classes)
plot_confusion_matrix(confnorm, labels=classes)
acc4= {}
acc_mod_snr = np.zeros( (len(classes),len(snrs)) )
i = 0
for snr in snrs:
        # Extract classes @ SNR
        test_SNRs = [lbl[x][1] for x in test_idx]
        test_X_i=X_test[np.where(np.array(test_SNRs) == snr)]
        test_X1_i = X1_test[np.where(np.array(test_SNRs) == snr)]
        test_X2_i = X2_test[np.where(np.array(test_SNRs) == snr)]
        test_Y_i = Y_test[np.where(np.array(test_SNRs) == snr)]


        test_Y_i_hat = model4.predict([test_X1_i,test_X2_i])

        confnorm_i,cor,ncor = calculate_confusion_matrix(test_Y_i,test_Y_i_hat,classes)
        acc4[snr] = 1.0 * cor / (cor + ncor)
        result = cor / (cor + ncor)

        plot_confusion_matrix(confnorm_i, labels=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM'])
        plt.show()  # Display the plot
        plt.savefig('figure3.png')  # Save the figure to a file

        acc_mod_snr[:,i] = np.round(np.diag(confnorm_i)/np.sum(confnorm_i,axis=1),3)
        i = i +1


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

num_mod = 11

plt.figure(figsize=(8, 6))
plt.xlabel("Signal to Noise Ratio")
plt.ylabel("Classification Accuracy")
plt.title("Classification Accuracy for Each Mod")

for i in range(0, num_mod):
    plt.plot(snrs, acc_mod_snr[i], label=classes[i])

# Add legend, grid, save, and show plot
plt.legend()
plt.grid()
plt.savefig('marah1.png')
plt.show()
plt.close()


In [ ]:
# Plot Accuracy vs. SNR
plt.figure(figsize=(10, 6))
plt.plot(snrs, list(map(lambda x: acc4[x], snrs)), marker='o', linestyle='-', color='b', label='Accuracy')

for snr, accuracy in zip(snrs, list(map(lambda x: acc4[x], snrs))):
    plt.annotate(f'{accuracy:.2f}',
                 (snr, accuracy),
                 textcoords="offset points",
                 xytext=(0,10),
                 ha='center')

plt.xlabel("Signal to Noise Ratio (SNR)")
plt.ylabel("Classification Accuracy")
plt.title("Classification Accuracy vs. SNR")
plt.grid(True)
plt.savefig('figure1.png')
plt.show()

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Reshape,GRU,LSTM
from tensorflow.keras.optimizers import Adam

# Assuming X_train, X_val, X_test, Y_train, Y_val, and Y_test are already loaded
# Normalize the data
X_train = X_train / np.max(np.abs(X_train), axis=(1,2), keepdims=True)
X_val = X_val / np.max(np.abs(X_val), axis=(1,2), keepdims=True)
X_test = X_test / np.max(np.abs(X_test), axis=(1,2), keepdims=True)

# Build the CNN model
model1= Sequential([
    Conv1D(64, kernel_size=3, activation='relu', input_shape=(128, 2)),
    MaxPooling1D(pool_size=2), Dropout(0.3),
    Conv1D(128, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),Dropout(0.3),
    Conv1D(256, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),Dropout(0.3),
    # temporal feature
    LSTM(units=128),
    Flatten(),
    Dense(512, activation='relu'), Dropout(0.3),
    Dense(11, activation='softmax')
])

# Compile the model
model1.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model1.load_weights('/content/drive/MyDrive/Colab Notebooks/CNN_LSTM_weights.h5')
# Train the model
#history = model.fit(X_train, Y_train, epochs=20, batch_size=64,
     #    validation_data=(X_val, Y_val))
# model.load_weights('/content/drive/MyDrive/project/CNN_weights.h5')

# Evaluate the model
test_loss, test_acc = model1.evaluate(X_test, Y_test)
print(f'Test accuracy: {test_acc}')

# Plot the training history
import matplotlib.pyplot as plt

#plt.figure(figsize=(12, 4))
#plt.subplot(1, 2, 1)
#plt.plot(history.history['loss'], label='Training Loss')
#plt.plot(history.history['val_loss'], label='Validation Loss')
#plt.xlabel('Epoch')
#plt.ylabel('Loss')
#plt.legend()
#plt.grid(True)

#plt.subplot(1, 2, 2)
#plt.plot(history.history['accuracy'], label='Training Accuracy')
#plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
#plt.xlabel('Epoch')
#plt.ylabel('Accuracy')
#plt.legend()
#plt.grid(True)

#plt.show()


In [ ]:
model1.summary()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import csv

model1.load_weights('/content/drive/MyDrive/Colab Notebooks/CNN_LSTM_weights.h5')
classes=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM']
plt.figure(figsize=(12, 10))  # Example size, adjust as needed
test_Y_hat = model1.predict([X_test], batch_size=64)
confnorm, _, _ = calculate_confusion_matrix(Y_test, test_Y_hat, classes)
plot_confusion_matrix(confnorm, labels=modtype)
acc1 = {}
acc_mod_snr = np.zeros( (len(classes),len(snrs)) )
i = 0
for snr in snrs:
        # Extract classes @ SNR
        test_SNRs = [lbl[x][1] for x in test_idx]
        test_X_i=X_test[np.where(np.array(test_SNRs) == snr)]
        test_Y_i = Y_test[np.where(np.array(test_SNRs) == snr)]
        test_Y_i_hat = model1.predict([test_X_i])

        confnorm_i,cor,ncor = calculate_confusion_matrix(test_Y_i,test_Y_i_hat,classes)
        acc1[snr] = 1.0 * cor / (cor + ncor)
        result = cor / (cor + ncor)

        plot_confusion_matrix(confnorm_i,
                      labels=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM'])
        plt.show()
        plt.close()

        acc_mod_snr[:,i] = np.round(np.diag(confnorm_i)/np.sum(confnorm_i,axis=1),3)
        i = i +1


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

num_mod = 11

plt.figure(figsize=(8, 6))
plt.xlabel("Signal to Noise Ratio")
plt.ylabel("Classification Accuracy")
plt.title("Classification Accuracy for Each Mod")

for i in range(0, num_mod):
    plt.plot(snrs, acc_mod_snr[i], label=classes[i])

# Add legend, grid, save, and show plot
plt.legend()
plt.grid()
plt.savefig('marah1.png')
plt.show()
plt.close()


In [ ]:
# Plot Accuracy vs. SNR
plt.figure(figsize=(10, 6))
plt.plot(snrs, list(map(lambda x: acc1[x], snrs)), marker='o', linestyle='-', color='b', label='Accuracy')

for snr, accuracy in zip(snrs, list(map(lambda x: acc1[x], snrs))):
    plt.annotate(f'{accuracy:.2f}',
                 (snr, accuracy),
                 textcoords="offset points",
                 xytext=(0,10),
                 ha='center')

plt.xlabel("Signal to Noise Ratio (SNR)")
plt.ylabel("Classification Accuracy")
plt.title("Classification Accuracy vs. SNR")
plt.grid(True)
plt.savefig('figure1.png')
plt.show()

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Reshape,GRU
from tensorflow.keras.optimizers import Adam

# Assuming X_train, X_val, X_test, Y_train, Y_val, and Y_test are already loaded
# Normalize the data
X_train = X_train / np.max(np.abs(X_train), axis=(1,2), keepdims=True)
X_val = X_val / np.max(np.abs(X_val), axis=(1,2), keepdims=True)
X_test = X_test / np.max(np.abs(X_test), axis=(1,2), keepdims=True)

# Build the CNN model
model2 = Sequential([
    Conv1D(64, kernel_size=3, activation='relu', input_shape=(128, 2)),
    MaxPooling1D(pool_size=2), Dropout(0.3),
    Conv1D(128, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),Dropout(0.3),
    Conv1D(256, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),Dropout(0.3),
    # Reshape(target_shape=((117,25)), name='reshape4'),
    # temporal feature
    GRU(units=128),
    Flatten(),
    Dense(512, activation='relu'), Dropout(0.3),
    Dense(11, activation='softmax')
])

# Compile the model
model2.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model2.summary()
model2.load_weights('/content/drive/MyDrive/Colab Notebooks/CNN_GRU_weights.h5')

# Train the model
#history = model.fit(X_train, Y_train, epochs=20, batch_size=64,
    #     validation_data=(X_val, Y_val))
# model.load_weights('/content/drive/MyDrive/project/CNN_weights.h5')

# Evaluate the model
#test_loss, test_acc = model2.evaluate(X_test, Y_test)




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import csv

model2.load_weights('/content/drive/MyDrive/Colab Notebooks/CNN_GRU_weights.h5')
classes=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM']
plt.figure(figsize=(12, 10))  # Example size, adjust as needed
test_Y_hat = model2.predict([X_test], batch_size=64)
confnorm, _, _ = calculate_confusion_matrix(Y_test, test_Y_hat, classes)
plot_confusion_matrix(confnorm, labels=classes)
acc2 = {}
acc_mod_snr = np.zeros( (len(classes),len(snrs)) )
i = 0
for snr in snrs:
        test_SNRs = [lbl[x][1] for x in test_idx]
        test_X_i=X_test[np.where(np.array(test_SNRs) == snr)]
        test_Y_i = Y_test[np.where(np.array(test_SNRs) == snr)]

        test_Y_i_hat = model2.predict([test_X_i])

        confnorm_i,cor,ncor = calculate_confusion_matrix(test_Y_i,test_Y_i_hat,classes)
        acc2[snr] = 1.0 * cor / (cor + ncor)
        result = cor / (cor + ncor)
        plot_confusion_matrix(confnorm_i,
                      labels=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM'])
        plt.show()
        plt.close()

        acc_mod_snr[:,i] = np.round(np.diag(confnorm_i)/np.sum(confnorm_i,axis=1),3)
        i = i +1


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

num_mod = 11

plt.figure(figsize=(8, 6))
plt.xlabel("Signal to Noise Ratio")
plt.ylabel("Classification Accuracy")
plt.title("Classification Accuracy for Each Mod")

for i in range(0, num_mod):
    plt.plot(snrs, acc_mod_snr[i], label=classes[i])

# Add legend, grid, save, and show plot
plt.legend()
plt.grid()
plt.savefig('marah1.png')
plt.show()
plt.close()


In [ ]:
# Plot Accuracy vs. SNR
plt.figure(figsize=(10, 6))
plt.plot(snrs, list(map(lambda x: acc2[x], snrs)), marker='o', linestyle='-', color='b', label='Accuracy')

for snr, accuracy in zip(snrs, list(map(lambda x: acc1[x], snrs))):
    plt.annotate(f'{accuracy:.2f}',
                 (snr, accuracy),
                 textcoords="offset points",
                 xytext=(0,10),
                 ha='center')

plt.xlabel("Signal to Noise Ratio (SNR)")
plt.ylabel("Classification Accuracy")
plt.title("Classification Accuracy vs. SNR")
plt.grid(True)
plt.savefig('figure1.png')
plt.show()

In [ ]:
plt.plot(snrs, list(map(lambda x: acc4[x], snrs)), label=' CNN')
plt.plot(snrs, list(map(lambda x: acc2[x], snrs)), label='CNN+GRU')
plt.plot(snrs, list(map(lambda x: acc1[x], snrs)), label='CNN+LSTM')
plt.plot(snrs, list(map(lambda x: acc[x], snrs)), label='newCNN+GRU')


plt.legend()
plt.grid(True)
plt.xlabel("Signal to Noise Ratio")
plt.ylabel("Classification Accuracy")
plt.title(" Classification Accuracy vs Signal to Noise Ratio")
plt.tight_layout()
plt.savefig('figure1.png')  # Save the figure to a file
plt.show()  # Display the plot
